# **Adult Income Classification**

## **Obtener la data**

In [ ]:
from sklearn.datasets import fetch_openml
import pandas as pd

adult = fetch_openml(name='adult', version=2, as_frame=True)
X = adult.data.copy()
y = adult.target.copy()

print(adult.DESCR[:2000])

## **Variables**

In [ ]:
X.columns.to_list()

In [ ]:
X.head()

Asignamos el target binario a formato 0 / 1

In [ ]:
y = (y == '>50K').astype(int)
y.value_counts(normalize=True)

## **Feature Engineer**

In [ ]:
X.head()

### **Distribucion de target: y**

In [ ]:
y.hist()

In [ ]:
X.info()

## **EDA**

Analisis de variables, relaciones importantes y creación de nuevas variables

In [ ]:
X.describe(include='all').T.head(15)

In [ ]:
X['age'].hist(bins=30)

In [ ]:
X['target'] = y

In [ ]:
X[X['age'] > 50]['target'].hist()

### **Crear variables**

In [ ]:
import numpy as np

X['capital_balance'] = X['capital-gain'] - X['capital-loss']
X['hours_per_week_group'] = pd.cut(
    X['hours-per-week'],
    bins=[0, 25, 40, 60, 100],
    labels=['part_time', 'standard', 'extended', 'very_high'],
    include_lowest=True
)
X['is_married'] = np.where(
    X['marital-status'].astype(str).str.contains('Married', case=False, na=False),
    1,
    0
)
X['has_capital_change'] = np.where(
    (X['capital-gain'] > 0) | (X['capital-loss'] > 0),
    1,
    0
)
X['education_num_per_age'] = X['education-num'] / X['age']
X[['capital_balance', 'hours_per_week_group', 'is_married', 'has_capital_change', 'education_num_per_age']].head()

In [ ]:
X.replace([np.inf, -np.inf], np.nan, inplace=True)

## **Feature Engineer**

In [ ]:
len(X)

In [ ]:
X.info()

In [ ]:
X.isna().sum().sort_values(ascending=False).head(15)

## 1. Missing data fill

In [ ]:
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator

In [ ]:
imputer_constant = SimpleImputer(strategy='constant', fill_value='missing')

In [ ]:
X.head()

In [ ]:
X_imputed_demo = X.copy()
cat_demo_cols = X_imputed_demo.select_dtypes(include='object').columns.tolist() + ['hours_per_week_group']
num_demo_cols = X_imputed_demo.select_dtypes(exclude='object').drop(columns='target').columns.tolist()

X_imputed_demo[cat_demo_cols] = imputer_constant.fit_transform(X_imputed_demo[cat_demo_cols])
X_imputed_demo[num_demo_cols] = SimpleImputer(strategy='median').fit_transform(X_imputed_demo[num_demo_cols])
X_imputed_demo.head()

## One-hot or codification strings

In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

nominal_cols = ['workclass', 'education', 'occupation']

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first')
nominal_encoded = ohe.fit_transform(X[nominal_cols])
nominal_encoded[:3]

In [ ]:
ohe.get_feature_names_out(nominal_cols)

In [ ]:
nominal_df = pd.DataFrame(
    nominal_encoded,
    columns=ohe.get_feature_names_out(nominal_cols),
    index=X.index
)
nominal_df.head()

In [ ]:
pd.concat([X[nominal_cols].head(), nominal_df.head()], axis=1)

In [ ]:
## Lo de arriba es explicativo, luego lo metemos dentro del pipeline
X.head()

## Escalamiento

In [ ]:
import matplotlib.pyplot as plt

X.select_dtypes(exclude='object').drop(columns='target').hist(figsize=(14, 10), bins=30)
plt.suptitle('Distribuciones originales')
plt.tight_layout()
plt.show()

In [ ]:
scaler = StandardScaler()
X_scaled_demo = scaler.fit_transform(X.select_dtypes(exclude='object').drop(columns='target').fillna(0))
X_scaled_demo[:3]

In [ ]:
X.head()

## Implementacion punta a punta

In [ ]:
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [ ]:
X.columns

In [ ]:
cat_cols = [
    'workclass', 'education', 'marital-status', 'occupation', 'relationship',
    'race', 'sex', 'native-country', 'hours_per_week_group'
]

num_cols = [col for col in X.columns if col not in cat_cols + ['target']]

numeric_simple = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_simple = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

In [ ]:
preprocess_simple = ColumnTransformer(transformers=[
    ('num', numeric_simple, num_cols),
    ('cat', categorical_simple, cat_cols)
])

In [ ]:
model_simple = Pipeline(steps=[
    ('prep', preprocess_simple),
    ('clf', LogisticRegression(max_iter=3000))
])

In [ ]:
X_dropped = X.drop(columns='target')
y = X['target']

X_train, X_test, y_train, y_test = train_test_split(
    X_dropped,
    y,
    test_size=0.25,
    stratify=y,
    random_state=12345
)

In [ ]:
X_train.head()

In [ ]:
X_test.head()

In [ ]:
model_simple.fit(X_train, y_train)

In [ ]:
model_simple.predict(X_test)

In [ ]:
model_simple.predict_proba(X_test)

In [ ]:
model_simple.predict_proba(X_test)[:, 1]

In [ ]:
pred_proba_simple = model_simple.predict_proba(X_test)[:, 1]
print('AUC simple imputer:', round(roc_auc_score(y_test, pred_proba_simple), 4))

# **1. Interpretacion Metricas**

In [ ]:
y_pred = model_simple.predict(X_test)
y_pred_proba = model_simple.predict_proba(X_test)[:, 1]

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay
)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print('Accuracy:', round(acc, 4))
print('Precision:', round(prec, 4))
print('Recall:', round(rec, 4))
print('F1:', round(f1, 4))
print('AUC:', round(auc, 4))

RocCurveDisplay.from_predictions(y_test, y_pred_proba)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=['Real 0', 'Real 1'], columns=['Pred 0', 'Pred 1'])
cm_df

In [ ]:
print(classification_report(y_test, y_pred))

# **2. Oversampling**

In [ ]:
try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE
except ImportError:
    %pip install imbalanced-learn -q
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE

model_oversampled = ImbPipeline(steps=[
    ('prep', preprocess_simple),
    ('oversample', SMOTE(random_state=52)),
    ('clf', LogisticRegression(max_iter=3000))
])

model_oversampled.fit(X_train, y_train)

In [ ]:
y_pred_oversampled = model_oversampled.predict(X_test)
y_pred_proba_oversampled = model_oversampled.predict_proba(X_test)[:, 1]

In [ ]:
acc = accuracy_score(y_test, y_pred_oversampled)
prec = precision_score(y_test, y_pred_oversampled)
rec = recall_score(y_test, y_pred_oversampled)
f1 = f1_score(y_test, y_pred_oversampled)
auc = roc_auc_score(y_test, y_pred_proba_oversampled)

print('Accuracy:', round(acc, 4))
print('Precision:', round(prec, 4))
print('Recall:', round(rec, 4))
print('F1:', round(f1, 4))
print('AUC:', round(auc, 4))

In [ ]:
cm = confusion_matrix(y_test, y_pred_oversampled)
cm_df = pd.DataFrame(cm, index=['Real 0', 'Real 1'], columns=['Pred 0', 'Pred 1'])
cm_df

In [ ]:
print(classification_report(y_test, y_pred_oversampled))

# **3. Implementar CV**

In [ ]:
from sklearn.model_selection import cross_val_score, cross_val_predict, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=52)

scores = cross_val_score(
    model_simple,
    X_train,
    y_train,
    cv=cv,
    scoring='roc_auc'
)

print('CV scores:', scores)
print('Mean AUC:', scores.mean())

In [ ]:
y_pred_kfold = cross_val_predict(model_simple, X_test, y_test, cv=cv)
y_pred_kfold[:10]

In [ ]:
print(classification_report(y_test, y_pred_kfold))

In [ ]:
print('1er modelo')
print(classification_report(y_test, y_pred))

# **4. Feature Importance**

Extraemos los coeficientes del modelo

In [ ]:
model_simple.fit(X_train, y_train)

feature_names = model_simple.named_steps['prep'].get_feature_names_out()
coefficients = model_simple.named_steps['clf'].coef_[0]

feature_importance = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients,
    'abs_coefficient': np.abs(coefficients)
}).sort_values('abs_coefficient', ascending=False)

feature_importance.head(20)

# **4.1 Shap values**

In [ ]:
try:
    import shap
except ImportError:
    %pip install shap -q
    import shap

X_train_transformed = model_simple.named_steps['prep'].transform(X_train)
X_test_transformed = model_simple.named_steps['prep'].transform(X_test)
feature_names = model_simple.named_steps['prep'].get_feature_names_out()

if hasattr(X_test_transformed, 'toarray'):
    X_test_transformed_dense = X_test_transformed.toarray()
else:
    X_test_transformed_dense = X_test_transformed

explainer = shap.LinearExplainer(
    model_simple.named_steps['clf'],
    X_test_transformed_dense,
    feature_names=feature_names
)
shap_values = explainer(X_test_transformed_dense)

## Summary plot

In [ ]:
shap.summary_plot(shap_values.values, X_test_transformed_dense, feature_names=feature_names)

## Bar plot

In [ ]:
shap.summary_plot(shap_values.values, X_test_transformed_dense, feature_names=feature_names, plot_type='bar')

## Local explanation

In [ ]:
row_idx = 0
shap.plots.waterfall(shap_values[row_idx])

# **5. Jugar con otros modelos**

En esta sección vamos a probar modelos distintos a la regresión logística para comparar desempeño.

## **5.1 Modelos a probar**

### **Random Forest**
Random Forest combina muchos árboles de decisión entrenados sobre muestras distintas.

### **XGBoost**
XGBoost suele capturar relaciones no lineales y combinaciones complejas entre variables.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

try:
    from xgboost import XGBClassifier
except ImportError:
    %pip install xgboost -q
    from xgboost import XGBClassifier

from sklearn.model_selection import cross_val_score, StratifiedKFold

## **5.2 Función para evaluar modelos**

In [ ]:
def evaluate_model(name, fitted_model, X_eval, y_eval, cv_scores=None):
    y_pred_local = fitted_model.predict(X_eval)
    y_pred_proba_local = fitted_model.predict_proba(X_eval)[:, 1]

    results = {
        'model': name,
        'accuracy': accuracy_score(y_eval, y_pred_local),
        'precision': precision_score(y_eval, y_pred_local),
        'recall': recall_score(y_eval, y_pred_local),
        'f1': f1_score(y_eval, y_pred_local),
        'auc_test': roc_auc_score(y_eval, y_pred_proba_local),
    }

    if cv_scores is not None:
        results['auc_cv_mean'] = cv_scores.mean()
        results['auc_cv_std'] = cv_scores.std()

    return results

## **5.3 Random Forest**

In [ ]:
rf_model = Pipeline(steps=[
    ('prep', preprocess_simple),
    ('clf', RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_split=20,
        min_samples_leaf=10,
        max_features='sqrt',
        random_state=52,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)
rf_cv_scores = cross_val_score(rf_model, X_train, y_train, cv=cv, scoring='roc_auc')
rf_results = evaluate_model('Random Forest', rf_model, X_test, y_test, rf_cv_scores)
rf_results

## **5.4 XGBoost**

In [ ]:
xgb_model = Pipeline(steps=[
    ('prep', preprocess_simple),
    ('clf', XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=52
    ))
])

xgb_model.fit(X_train, y_train)
xgb_cv_scores = cross_val_score(xgb_model, X_train, y_train, cv=cv, scoring='roc_auc')
xgb_results = evaluate_model('XGBoost', xgb_model, X_test, y_test, xgb_cv_scores)
xgb_results

## **5.5 Comparación contra el primer modelo**

In [ ]:
baseline_results = evaluate_model('Logistic Regression', model_simple, X_test, y_test, scores)
comparison_df = pd.DataFrame([
    baseline_results,
    rf_results,
    xgb_results
]).round(4)

comparison_df.sort_values('auc_test', ascending=False)

## **6. Optimización con Optuna**

Ahora usamos **Optuna** para buscar mejores hiperparámetros.

In [ ]:
try:
    import optuna
except ImportError:
    %pip install optuna -q
    import optuna

optuna.__version__

### **6.1 Optuna para Random Forest**

In [ ]:
def objective_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 150, 500),
        'max_depth': trial.suggest_int('max_depth', 4, 18),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 30),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 15),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2'])
    }

    rf_trial_model = Pipeline(steps=[
        ('prep', preprocess_simple),
        ('clf', RandomForestClassifier(**params, random_state=52, n_jobs=-1))
    ])

    scores_trial = cross_val_score(rf_trial_model, X_train, y_train, cv=cv, scoring='roc_auc')
    return scores_trial.mean()

study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(objective_rf, n_trials=15)
study_rf.best_params

In [ ]:
rf_best_model = Pipeline(steps=[
    ('prep', preprocess_simple),
    ('clf', RandomForestClassifier(**study_rf.best_params, random_state=52, n_jobs=-1))
])

rf_best_model.fit(X_train, y_train)
rf_best_cv_scores = cross_val_score(rf_best_model, X_train, y_train, cv=cv, scoring='roc_auc')
rf_best_results = evaluate_model('Random Forest + Optuna', rf_best_model, X_test, y_test, rf_best_cv_scores)
rf_best_results

### **6.2 Optuna para XGBoost**

In [ ]:
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 150, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 8),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0.0, 3.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10.0, log=True),
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'random_state': 52
    }

    xgb_trial_model = Pipeline(steps=[
        ('prep', preprocess_simple),
        ('clf', XGBClassifier(**params))
    ])

    scores_trial = cross_val_score(xgb_trial_model, X_train, y_train, cv=cv, scoring='roc_auc')
    return scores_trial.mean()

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=15)
study_xgb.best_params

In [ ]:
xgb_best_params = study_xgb.best_params.copy()
xgb_best_params.update({
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'random_state': 52
})

xgb_best_model = Pipeline(steps=[
    ('prep', preprocess_simple),
    ('clf', XGBClassifier(**xgb_best_params))
])

xgb_best_model.fit(X_train, y_train)
xgb_best_cv_scores = cross_val_score(xgb_best_model, X_train, y_train, cv=cv, scoring='roc_auc')
xgb_best_results = evaluate_model('XGBoost + Optuna', xgb_best_model, X_test, y_test, xgb_best_cv_scores)
xgb_best_results

## **6.3 Comparación final**

In [ ]:
final_comparison_df = pd.DataFrame([
    baseline_results,
    rf_results,
    xgb_results,
    rf_best_results,
    xgb_best_results
]).round(4)

final_comparison_df.sort_values('auc_test', ascending=False)

## **6.4 Conclusión sugerida para explicar en clase o en una presentación**

- La regresión logística funciona bien como baseline porque es interpretable.
- Random Forest y XGBoost permiten capturar relaciones no lineales.
- La validación con CV evita quedarse solo con un split.
- SHAP ayuda a explicar qué variables empujan la predicción.
- Optuna permite mejorar hiperparámetros de forma ordenada.